In [1]:
import numpy as np
import pandas as pd
from scipy.interpolate import interp1d
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")
from matplotlib.ticker import MaxNLocator
import time

#### <font color=red> Section I: Concentration parameter and vonmises profile function</font>

In [2]:
def duty_cycle_to_kappa(D: float) -> float:
    """
    Convert FWHM duty cycle D to von Mises concentration κ.

    The von Mises FWHM satisfies  2·arccos(1 - log2/κ) = 2π·D,
    which inverts to κ = log(2) / (2·sin²(πD/2)).
    """
    return np.log(2.0) / (2.0 * np.sin(np.pi * D / 2.0) ** 2)



def von_mises_profile(phi: np.ndarray, kappa: float) -> np.ndarray:
    """
    ρ(φ) = exp(κ·(cos φ − 1))   — Eq. (2).

    Peak at φ=0, falls to ½ at |φ| = arccos(1 - log2/κ).
    Equivalent to a wrapped Gaussian in the large-κ limit.
    """
    vonmis_rho = np.exp(kappa * (np.cos(phi) - 1.0))
    return vonmis_rho

In [3]:
def profile_harmonics(kappa: float, n_harmonics) -> np.ndarray:
    """
    Fourier coefficients ρ_n of the detrended von Mises profile.

    Evaluates ρ on a 4096-point grid, removes the DC term (detrending, §III),
    and returns coefficients for n = -n_harmonics … +n_harmonics packed into
    a real-valued array of length 2·n_harmonics + 1:

        result[n + n_harmonics]  =  ρ_n      (complex)

    Detrending (ρ_0 = 0) is equivalent to high-pass filtering the data before
    the search — it removes any DC offset from the timestream.
    """
    N_grid = 4
    phi    = np.linspace(0.0, 2.0 * np.pi, N_grid, endpoint=False)
    rho    = von_mises_profile(phi, kappa)
    rho    = rho - rho.mean()                   # remove DC
    rho_fft = np.fft.fft(rho) / N_grid
    return np.array([rho_fft[n % N_grid] for n in range(-n_harmonics, n_harmonics + 1)])
    # return rho

### <font color=red>Section V:  Breaking "down data_fft()"  function. (Finding interpolating polynomial function using frequency values from Fourier transformed data)</font>
In this section I will try to breakdown in detail and step by step what this function(data_fft) does. This funftion is eqn 15 in the paper and the goal is to code that function to return the fourier transform of data. 

In [4]:
def data_fft(data: np.ndarray, ts: float, zero_pad_factor: int = 2):
    """
    Zero-pad and FFT the timestream, returning a cubic-interpolating function
    d_tilde(omega) that evaluates the Fourier transform at any frequency.

    Phase convention (Eq. 15):
        d̃(ω) = ts · Σ_k  d_k · exp(i·ω·(k·ts + ts/2 − T/2))

    The extra phase  exp(−i·ω·T/2)  centres the phase reference at t = T/2.
    This makes d̃(ω) vary more slowly with ω, improving cubic interpolation.

    Returns
    -------
    d_tilde : callable  omega -> complex value of d̃(omega)
    """
    N      = len(data)
    T      = N * ts
    N_pad  = zero_pad_factor * N

    padded = np.zeros(N_pad)
    padded[:N] = data

    raw_fft   = np.fft.fft(padded) * ts
    omega_fft = 2.0 * np.pi * np.fft.fftfreq(N_pad, d=ts)

    # Apply the centring phase shift  exp(i·ω·(ts/2 − T/2))
    phase_shift = np.exp(1j * omega_fft * (ts / 2.0 - T / 2.0))
    d_fft       = raw_fft * phase_shift

    # Sort by frequency for interpolation
    order      = np.argsort(omega_fft)
    omega_sort = omega_fft[order]
    d_sort     = d_fft[order]

    interp_re = interp1d(omega_sort, d_sort.real, kind="cubic",
                         bounds_error=False, fill_value=0.0)
    interp_im = interp1d(omega_sort, d_sort.imag, kind="cubic",
                         bounds_error=False, fill_value=0.0)

    def d_tilde(omega: float) -> complex:
        return interp_re(omega) + 1j * interp_im(omega)

    return d_tilde

### <font color=red>Section VI:  Breaking down "normalization()" function </font>

This function will calculate normalization factor. It is basically evaluating equation 13 in the paper. 

$$ A(\omega_{c}) \approx \eta^{-2}T\sum_{n} |\rho_{n}|^{2} j_{0}\left(\frac{n\omega_{c}t_{s}}{2} \right)^{2} $$

Last term $j_{0}$ is a sinc function. This normalization takes vonmise profile($\rho$), angular frequency($\omega$), data resolution, total timelength of data, noise parameter($\eta$) and number of harmonics considered as input. This function will return a single float numer as normalization factor. In the paper normalization is done to SNR value of 1.

As x-->0, we know,

$$  \lim_{x\to 0} \frac{sin(x)}{x}   = 1  $$
so for x<1e-12 value of sinc function is returned 1. Normalization factor is calculated for each center frequency $\omega_{c}$

In [5]:
def normalisation(
    rho_n:   np.ndarray,
    omega_c: float,
    ts:      float,
    T:       float,
    eta:     float,
    n_harmonics: int,
) -> float:
    """
    A(ωc) ≈ η⁻² · T · Σ_{n≠0} |ρ_n|² · j₀(n·ωc·ts/2)²   — Eq. (13).

    j₀(x) = sin(x)/x  is the Fourier transform of a rectangular window of
    width ts.  It corrects for the fact that the timestream is boxcar-averaged
    over each sample: high harmonics are attenuated by the finite bin width.

    The approximation uses  ∫ e^{i(m+n)ωt} dt ≈ T·δ_{m,−n}  (large T limit).
    """
    def j0(x: float) -> float:                                # Sinc function evaluation
        return 1.0 if abs(x) < 1e-12 else np.sin(x) / x       # 
    total = 0.0
    for n in range(-n_harmonics, n_harmonics + 1):
        if n == 0:
            continue
        rho  = rho_n[n + n_harmonics]                         # Vonmise profile
        corr = j0(n * omega_c * ts / 2.0)
        total += abs(rho) ** 2 * corr ** 2

    return (eta ** -2) * T * total

### <font color=red>Section VII:  Breaking down phase and frequency grid calculations </font>

These two functions are simple function to calculate trial phase and frequency grid values. These values will be used to calculate the detection statistics and the one with the highest value should give optimal phase and fequency. Frequency resolution is given by D/T, where D is duty cycle and T is the total time length of the data. Minimum and maximum value of frequency we want to look into is supplied as an input. Number of frequency values is at least 1 or $f_{max}$ - $f_{min}$ divided by frequency resolution and the resulting value is rounded to nearest integer. In practice, we need to make educated choice of max and min value of frequency to be given as input. 
Note: Probably, the way we decided frequency range in PhaseI of our work should work. 

In the second function "trial_phi_grid()" number of trial phases is chosen based on the duty cycle(D) preferred. Number of phase factor is supplied by the user. Here, 2 is chosen. So for duty cycle of 0.1, N_phi = 20 and hence 20 different phase values linearly spaced between 0 and $2\pi$ is obtained. If we want finer grid we can increase the phase factor. 

In [6]:
def trial_omega_gridI(
    omega_min: float,
    omega_max: float,
    D:         float,
    T:         float,
) -> np.ndarray:
    """
    Frequency grid with spacing Δω = D/T.

    Δω = D/T ensures adjacent trial frequencies differ by less than one
    resolution element in the detection statistic Ê(ωc, Φc).
    """
    T_c  = T
    d_omega = 10*(D/T_c)
    n_steps = max(1, int((omega_max - omega_min) / d_omega))      # Ensures minimum of 1 frequency value
    return np.linspace(omega_min, omega_max, n_steps)

def trial_omega_gridII(fmin, fmax, totalrange):
    return np.linspace(fmin, fmax, totalrange)
    
def trial_omega_gridIII(omega_min: float, omega_max: float, delta_t: float) -> np.ndarray:
    w_omega = 2*int(2*np.pi*(1/delta_t))
    # delta_omega = (omega_max - omega_min)/w_omega
    return np.linspace(omega_min, omega_max, w_omega)


    
def trial_phi_grid(D: float, n_phi_factor: float = 2.0) -> np.ndarray:
    """
    Phase grid with N_Φ = n_phi_factor / D points over [0, 2π).

    N_Φ ≈ 2/D is the conservative default from Appendix A.
    Finer grids (larger n_phi_factor) reduce suboptimality at the cost of
    more FFT evaluations.
    """
    N_phi = max(8, int(n_phi_factor / D))       # Ensures N_phi is at least 8.  
    return np.linspace(0.0, 2.0 * np.pi, N_phi, endpoint=False)

### <font color=red>Section VIII:  Breaking down detection statistics function</font>


In [7]:
def eb_at_omega(
    omega_c:    float,
    d_tilde,                # callable: omega -> complex
    rho_n:      np.ndarray,
    A:          float,
    ts:         float,
    eta:        float,
    n_harmonics: int,
) -> np.ndarray:
    """
    Ê(ωc, Φc) for all trial phases at a fixed trial frequency ωc — Eq. (16).

    Ê(ωc, Φc) = A(ωc)^{-1/2} · η^{-2} · Σ_n  ρ_n · j₀(n·ωc·ts/2) · d̃(n·ωc) · e^{i·n·Φc}

    The sum over harmonics n is the discrete Fourier series in Φc, so a
    single IFFT over harmonic index n gives Ê at all N_Φ phase values at once.

    Steps:
        1. Build a complex array  F[n]  =  ρ_n · j₀(n·ωc·ts/2) · d̃(n·ωc)
        2. IFFT of F  →  Ê(ωc, Φc)  for all Φc
        3. Scale by  A^{-1/2} · η^{-2}

    Returns shape (N_phi,) where N_phi is determined by the FFT size chosen
    to cover all harmonics from -n_harmonics to +n_harmonics.
    """
    def j0(x: float) -> float:
        return 1.0 if abs(x) < 1e-12 else np.sin(x) / x

    N_fft = 2 * n_harmonics   # one bin per harmonic, positive and negative

    # F is indexed 0 … N_fft-1 using the standard FFT frequency ordering:
    # bin k corresponds to harmonic  k  (for k <= N_fft//2)
    # and harmonic  k - N_fft       (for k > N_fft//2)
    F = np.zeros(N_fft, dtype=complex)

    for n in range(-n_harmonics, n_harmonics + 1):
        if n == 0:
            continue
        rho  = rho_n[n + n_harmonics]
        corr = j0(n * omega_c * ts / 2.0)
        d    = d_tilde(n * omega_c)
        k    = n % N_fft          # map harmonic n to FFT bin index
        F[k] += rho * corr * d

    # IFFT over harmonic index  →  Ê as function of phase
    # Factor of N_fft because np.ifft divides by N
    Eb_complex = np.fft.ifft(F) * N_fft

    # Take real part (imaginary part is numerical noise) and normalise
    if A < 1e-30:
        return np.zeros(N_fft)

    return Eb_complex.real / (np.sqrt(A) * eta ** 2)

In [8]:
def fast_const_period_algorithm(trial_data, res_ts, w_min, w_max,):
    def j0_sinc(x: float) -> float:
            return 1.0 if abs(x) < 1e-12 else np.sin(x) / x

    N_tot = len(trial_data)
    total_T = N_tot*res_ts
    # true_w = 2*np.pi*true_freq
    dty_ccl = 0.1

    # w_values = trial_omega_gridI(w_min, w_max, dty_ccl, total_T)
    wval_trlgrdII = trial_omega_gridIII(w_min, w_max, res_ts)
    # print("Frequency spacing",wval_trlgrdII[1] - wval_trlgrdII[0])
    # print("Size of freq grid", len(wval_trlgrdII))
    # print('Freq spacing with trial grid I',w_values[1] - w_values[0])
    # print('Freq spacing with trial grid II',wval_trlgrdII[1] - wval_trlgrdII[0])

    num_harmonic = 2
    nfft_factor = num_harmonic*2
    eta_noiseval = 1
    phase_grid_values = np.linspace(0, 2*np.pi, 2*num_harmonic, endpoint=False)   #Phase gridded between 0 to 2*pi

    empty_cmplx_ary = np.zeros(nfft_factor, dtype=complex)
    tilde_d = data_fft(trial_data, res_ts, 2)
    concn_param = duty_cycle_to_kappa(dty_ccl)
    prof_rho = profile_harmonics(concn_param, num_harmonic)

    det_statistics = []
    for ind1 in range(0, len(wval_trlgrdII)):
        w_freq_val = wval_trlgrdII[ind1]
        normalization_val = normalisation(prof_rho, w_freq_val, res_ts, total_T, eta_noiseval, num_harmonic)    

        for ind2 in range(-num_harmonic, num_harmonic+1):
            if ind2 == 0:
                continue
            dtilde_fft = tilde_d(ind2*w_freq_val)
            j0_bxcr = j0_sinc(ind2*w_freq_val*(res_ts/2))
            rho_prof = prof_rho[ind2 + num_harmonic]
            kindx = ind2%nfft_factor
            empty_cmplx_ary[kindx]+= rho_prof*j0_bxcr*dtilde_fft
        cmplx_detstat = np.fft.ifft(empty_cmplx_ary)*nfft_factor
        det_stat = cmplx_detstat.real / (np.sqrt(normalization_val) * eta_noiseval ** 2)
        det_statistics.append(det_stat)


    detstatarrray = np.array(det_statistics)
    stat_index = np.unravel_index(np.argmax(detstatarrray), detstatarrray.shape)
    # print("Optimum index value  of search statitsitc", stat_index)
    optimum_value = detstatarrray[stat_index]
    optimum_period =  1/(wval_trlgrdII[stat_index[0]]/(2*np.pi))
    return optimum_period

### <font color='red'> Section VIII: Accessing simulated datafile and extracting timeseries data</font>

In [9]:
file_list = ['period_1.53sec_dty_cycl0.1_ang_accl_0_.txt',	'period_3.541sec_dty_cycl0.1_ang_accl_0_.txt',
             'period_1.785sec_dty_cycl0.1_ang_accl_0_.txt', 'period_2.12sec_dty_cycl0.1_ang_accl_0_.txt']# Below are actual period of corresponding timeseries data above in same order.
# actual_period = [1.53, 3.541, 1.785, 2.12]
predicted_period = []
time_taken = []

# We supply range of angular frequencies to look at. 
min_prd = 0.5
max_prd = 4.5
max_ang_freq = 2*np.pi*(1/min_prd)
min_ang_freq = 2*np.pi*(1/max_prd)
    
print("Predicted Period ")

for numindx in np.arange(len(file_list)):
    
    starttime = time.time()
    if numindx <=len(file_list):   
        filepath = "/Users/manojkhadka/research/claudeChatgptversion/mock_pulsar_data/"+str(file_list[numindx])
    # else:
    #     filepath = "/home/computemachine/Desktop/research/testdata/simulated_dataEF/sim_pulse_"+str(numindx)+ ".txt"
        # df = pd.read_csv(filepath, sep=r"\s+", skiprows=1)
        df = pd.read_csv(filepath, header=0, sep="\t\t")
        test_signal = df['intensity'] - min(df['intensity'])
        dat_resolution = df['time'][1] - df['time'][0]
        p_pred = fast_const_period_algorithm(test_signal, dat_resolution, min_ang_freq, max_ang_freq)
        endtime = time.time()
        timetaken = endtime - starttime
        time_taken.append(timetaken/60)   # Time taken by algorithm in minutes
        predicted_period.append(np.round(p_pred,4))
        print("\n" + str(np.round(p_pred, 4)))
    else:
        break


Predicted Period 

3.2935

3.2413

3.2732

3.2779


In [10]:
print("Avg time of detection for one data file ",np.mean(time_taken), 'mins')

Avg time of detection for one data file  20.13115372757117 mins


In [11]:
predicted_period

[np.float64(3.2935),
 np.float64(3.2413),
 np.float64(3.2732),
 np.float64(3.2779)]

In [12]:
Actual_period = [0.6669, 3.0918, 1.789, 1.2144, 6.3076, 1.1642, 1.5435, 0.9543, 0.8533, 1.6946, 0.5029, 2.5202, 1.2201, 3.2539, 0.8686, 0.5357, 1.0115,
0.5194, 0.8936, 2.5702, 0.5819, 1.2301, 0.5016, 0.82, 0.5507, 0.7988, 0.5253, 1.1477, 0.8414, 2.0636]

In [13]:
2.62/1.53

1.712418300653595